In [5]:
library(reticulate)

# Force reticulate to use system Python (bypasses virtualenv creation error)
use_python(Sys.which("python3"), required = TRUE)

# Install kagglehub using system pip instead of virtualenv
system("pip3 install --user kagglehub", ignore.stdout = TRUE, ignore.stderr = TRUE)

# Now import and download
kagglehub <- import("kagglehub")
path <- kagglehub$dataset_download("wordsforthewise/lending-club")
print(paste("Path to dataset files:", path))

# Find the CSV file (it's usually in a subdirectory)
files <- list.files(path, recursive = TRUE, full.names = TRUE, pattern = "accepted.*\\.csv")
csv_path <- files[1]
print(paste("Found CSV:", csv_path))

# Load the data
data <- read.csv(csv_path, nrows = 50000)

# Quick check
print(dim(data))
head(data, 3)

[1] "Path to dataset files: /kaggle/input/lending-club"
[1] "Found CSV: /kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz"
[1] 50000   151


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,⋯,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
,<int>,<lgl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,⋯,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,68407277,NA,3600,3600,3600,36 months,13.99,123.03,C,C4,⋯,NA,NA,Cash,N,,,,NA,NA,NA
2,68355089,NA,24700,24700,24700,36 months,11.99,820.28,C,C1,⋯,NA,NA,Cash,N,,,,NA,NA,NA
3,68341763,NA,20000,20000,20000,60 months,10.78,432.66,B,B4,⋯,NA,NA,Cash,N,,,,NA,NA,NA


In [6]:
# Keep only relevant columns
cols <- c('loan_status','loan_amnt','int_rate','annual_inc', 'dti','fico_range_low','grade','emp_length','home_ownership')
df <- data[, cols]

# Create binary target: 1 = Default, 0 = Paid
df$default <- ifelse(df$loan_status == 'Charged Off', 1, 0)

# Remove missing values
df <- na.omit(df)

# Convert int_rate from character '13.5%' to numeric 13.5
df$int_rate <- as.numeric(gsub('%', '', df$int_rate))

cat('Rows after cleaning:', nrow(df), '\n')
cat('Default rate:', round(mean(df$default)*100, 2), '%\n')

Rows after cleaning: 49999 
Default rate: 18.05 %


In [7]:
# Summary statistics
summary(df[, c('loan_amnt','annual_inc','int_rate','fico_range_low','dti')])

# Default rate by loan grade
default_by_grade <- aggregate(default ~ grade, data=df, FUN=mean)
default_by_grade$default_pct <- round(default_by_grade$default * 100, 2)
print(default_by_grade)

   loan_amnt       annual_inc         int_rate     fico_range_low 
 Min.   : 1000   Min.   :   1770   Min.   : 5.32   Min.   :660.0  
 1st Qu.: 8000   1st Qu.:  48000   1st Qu.: 9.17   1st Qu.:670.0  
 Median :14000   Median :  66000   Median :11.99   Median :685.0  
 Mean   :15019   Mean   :  79195   Mean   :12.23   Mean   :694.4  
 3rd Qu.:20000   3rd Qu.:  95000   3rd Qu.:14.48   3rd Qu.:710.0  
 Max.   :35000   Max.   :9000000   Max.   :28.99   Max.   :845.0  
      dti        
 Min.   :  0.00  
 1st Qu.: 12.68  
 Median : 18.82  
 Mean   : 19.34  
 3rd Qu.: 25.61  
 Max.   :999.00  

  grade    default default_pct
1     A 0.05138294        5.14
2     B 0.12552411       12.55
3     C 0.19791954       19.79
4     D 0.29136952       29.14
5     E 0.35640732       35.64
6     F 0.44287159       44.29
7     G 0.48648649       48.65


In [8]:
# Q: Is income significantly different between defaulters and non-defaulters?
defaulters <- df$annual_inc[df$default == 1]
non_defaulters <- df$annual_inc[df$default == 0]
result <- t.test(defaulters, non_defaulters)
print(result)

# Interpretation:
# If p-value < 0.05, income IS significantly different between groups
# This supports using income as a predictor of default


	Welch Two Sample t-test

data:  defaulters and non_defaulters
t = -9.4632, df = 19800, p-value < 2.2e-16
alternative hypothesis: true difference in means is not equal to 0
95 percent confidence interval:
 -10291.998  -6760.062
sample estimates:
mean of x mean of y 
 72208.04  80734.07 



In [9]:
# This is the core statistical model
model_r <- glm(default ~ loan_amnt + int_rate + annual_inc + dti + fico_range_low, data = df, family = binomial(link = 'logit'))
summary(model_r)

# Odds Ratios — what professors expect you to interpret
exp(coef(model_r))

# Model predictions
df$pred_prob <- predict(model_r, type = 'response')
df$pred_class <- ifelse(df$pred_prob > 0.5, 1, 0)

# Accuracy
accuracy_r <- mean(df$pred_class == df$default)
cat('R Logistic Regression Accuracy:', round(accuracy_r*100, 2), '%\n')


Call:
glm(formula = default ~ loan_amnt + int_rate + annual_inc + dti + 
    fico_range_low, family = binomial(link = "logit"), data = df)

Coefficients:
                 Estimate Std. Error z value Pr(>|z|)    
(Intercept)     6.798e-01  3.602e-01   1.887 0.059156 .  
loan_amnt       2.745e-06  1.610e-06   1.705 0.088106 .  
int_rate        1.365e-01  3.144e-03  43.409  < 2e-16 ***
annual_inc     -1.081e-06  3.040e-07  -3.555 0.000378 ***
dti             1.274e-02  1.408e-03   9.053  < 2e-16 ***
fico_range_low -6.072e-03  5.048e-04 -12.029  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 47221  on 49998  degrees of freedom
Residual deviance: 43710  on 49993  degrees of freedom
AIC: 43722

Number of Fisher Scoring iterations: 5


(Intercept)      loan_amnt       int_rate     annual_inc            dti 
     1.9734326      1.0000027      1.1462531      0.9999989      1.0128260 
fico_range_low 
     0.9939464

R Logistic Regression Accuracy: 81.95 %


In [10]:
# Compute PD (Probability of Default) by income group
df$income_group <- cut(df$annual_inc,
  breaks = quantile(df$annual_inc, probs=seq(0,1,0.2), na.rm=TRUE),
  labels = c('Very Low','Low','Medium','High','Very High'),
  include.lowest = TRUE)

pd_by_income <- aggregate(default ~ income_group, data=df, FUN=mean)
pd_by_income$PD_pct <- round(pd_by_income$default * 100, 2)
print(pd_by_income)

# Expected Loss formula — banks actually use this
# EL = PD x EAD x LGD
# EAD = Exposure at Default = loan amount
# LGD = Loss Given Default = assume 0.45 (industry standard)
df$EL <- df$pred_prob * df$loan_amnt * 0.45
cat('Average Expected Loss per loan: $', round(mean(df$EL), 2), '\n')

  income_group   default PD_pct
1     Very Low 0.2117000  21.17
2          Low 0.1935539  19.36
3       Medium 0.1904928  19.05
4         High 0.1652670  16.53
5    Very High 0.1401663  14.02
Average Expected Loss per loan: $ 1268.28 


In [11]:
library(jsonlite)

# Default rate by grade — for frontend chart
write(toJSON(default_by_grade), '/content/default_by_grade.json')

# Income vs default — for frontend chart
write(toJSON(pd_by_income), '/content/income_distribution.json')

# R model accuracy — for frontend metrics card
r_metrics <- data.frame(
  model = 'R Logistic Regression',
  accuracy = round(accuracy_r * 100, 2)
)
write(toJSON(r_metrics), '/content/r_metrics.json')

cat('All JSON files exported. Download them from the Files panel on the left.\n')
cat('Save them to your project results/ folder.\n')

All JSON files exported. Download them from the Files panel on the left.
Save them to your project results/ folder.
